# Measuring Areas and Distances


The accuracy of **distance** and **area** calculations and the construction of **buffer zones** depends directly on the **coordinate reference system (CRS)** being used.

When data is in a **geographic coordinate system** such as **WGS 84 (EPSG:4326)**, coordinates are expressed in **degrees**. This is suitable for **displaying features on a map**, but **not for measuring distances or areas directly**.

To get results in **metres** and **square metres**, the data must be reprojected into an appropriate **projected coordinate system** such as **UTM**.

In this section, we will cover:

- why data has to be **reprojected** before you measure anything, and how the **CRS shapes the numbers you get**;
- how to calculate **polygon areas**;
- how to measure **distances between features**.


## 0. Importing Libraries and Preparing the Data


### 0.1. Importing Libraries


In [ ]:
import osmnx as ox
import geopandas as gpd

from shapely.geometry import Point, LineString

# cache OSM responses on disk, so repeating a query does not hit the server again
ox.settings.cache_folder = "../../cache"

### 0.2. Preparing the Data


We start with the district boundary from OpenStreetMap.


In [ ]:
area_name = "Innere Stadt, Vienna, Austria"

admin_border = ox.geocode_to_gdf(area_name)
admin_border.explore(tiles="cartodbpositron")

Its coordinate reference system:


In [ ]:
admin_border.crs

Now the metro station entrances in the same district.


In [ ]:
tags = {"railway": "subway_entrance"}

metro = ox.features_from_place(area_name, tags)

metro.explore(tiles="cartodbpositron")

And their coordinate reference system:


In [ ]:
metro.crs

## 1. Measuring Area

In GeoPandas, every `GeoDataFrame` has a `.geometry` attribute that stores the **geometric objects** of the spatial dataset — points, lines, or polygons.

This attribute provides access to the geometry of each feature and enables various **spatial operations**.

One such operation is area calculation. The `.area` attribute returns the **area of each polygon feature**.

Remember that `.area` returns values **in the units of the coordinate reference system (CRS)**.


### 1.1. Calculating Area — Take 1

We add a column `"area_deg2"` to the `admin_border` GeoDataFrame and store the calculated areas in it.

In [ ]:
admin_border["area_deg2"] = admin_border.geometry.area

The result:


In [ ]:
admin_border[["name", "area_deg2"]]

At this point, you may notice that the area values are **unexpectedly small**. This is because the layer is still in a **geographic coordinate system**.

In geographic coordinate systems such as **WGS 84 (EPSG:4326)**, positions are defined in **degrees of latitude and longitude** rather than in linear units. As a result, area is calculated in **square degrees**.

Such values are difficult to interpret, because **degrees are not a unit of length** and their physical scale **varies with latitude**.

To calculate areas and distances correctly, the data must first be **reprojected** into a coordinate system with metric units, such as **UTM**.

> **Note the `UserWarning`** that may appear when calculating area.
> It indicates that the calculation is being performed on data in a **geographic coordinate system**, and the results may be unreliable. This warning is a reminder to **reproject the data** before taking measurements.


### 1.2. Reprojecting to UTM

Let's reproject the data into the appropriate **UTM zone** so that coordinates are expressed in **metres**.

First, we'll determine the suitable UTM CRS using the `.estimate_utm_crs()` method, then reproject the `admin_border` layer into it.


In [ ]:
utm_crs = admin_border.estimate_utm_crs()

admin_border_utm = admin_border.to_crs(utm_crs)

Confirm that the CRS has changed.


In [ ]:
admin_border_utm.crs

We can see that this CRS uses metres. Let's now recalculate the area.


### 1.3. Calculating Area — Take 2


Now the same for `admin_border_utm`, in a column called `"area_m2"`. The column with the degree-based values is still there, so the two can be compared side by side.

In [ ]:
admin_border_utm["area_m2"] = admin_border_utm.geometry.area

The result:


In [ ]:
admin_border_utm[["name", "area_deg2", "area_m2"]]

The values now look far more realistic, since area is being calculated in **square metres**.

Area values may sometimes be displayed in **scientific notation**, which is a compact way of representing large numbers.

A value of the form `a.bcde+X` means $a.bcde \times 10^{X}$.

For example:
`2.866748e+06` = $2.866748 \times 10^{6}$, or approximately 2.9 million.

This is simply an alternative way of writing a large number.


For convenience, the area can be converted to square kilometres (1 km² = 1,000,000 m²):


In [ ]:
admin_border_utm["area_km2"] = admin_border_utm["area_m2"] / 1_000_000
admin_border_utm[["name", "area_km2"]]

### 1.4. From a Polygon to a Point

Area is not the only thing a polygon will give you. The **centroid** is its geometric centre — the single point that stands in for the whole shape.

It comes up constantly: to label a polygon, to place a map on its study area, or to turn a layer of polygons into a layer of points so that distances can be measured to it at all. Like area, it is a geometric operation, so it must be computed on the **projected** layer — a centroid taken from degrees lands in the wrong place.

In [ ]:
center = admin_border_utm.geometry.centroid.iloc[0]

center_gdf = gpd.GeoDataFrame(geometry=[center], crs=utm_crs)

center_gdf.explore(tiles="cartodbpositron")

We will use this again in the [sixth module](../module_6/map_1.ipynb), where the centroid of the study area decides where the finished map opens.

## 2. Measuring Distances

The `.distance()` method calculates the distance between geometric features.
Suppose we are standing outside the **Wiener Staatsoper** — the opera house we built as a single point back in the [first module](../module_1/spData_1.ipynb) — and want to know where the nearest U-Bahn entrance is.

We will do this twice, as we did with the area: once on the data as it comes, and once after reprojecting. The failure is more interesting this time.


### 2.0. Preparing the Data


First, let's reproject the metro layer into the same CRS as the district boundary.


In [ ]:
metro_utm = metro.to_crs(utm_crs)

Then the point we are measuring from. Its coordinates come from `vienna_top_locations.csv`, the file used in the [previous section](projections_2.ipynb) — this is the same building that opened the first module.

We keep it in both systems, because the measurement below is done twice.

In [ ]:
opera = Point(16.368915, 48.202818)                       # Wiener Staatsoper, EPSG:4326
opera_utm = gpd.GeoSeries([opera], crs="EPSG:4326").to_crs(utm_crs).iloc[0]

### 2.1. Calculating Distances — Take 1

First, the measurement on the original data, still in degrees — both the entrances and the opera are in EPSG:4326 as they came.

In [ ]:
metro["distance_deg"] = metro.geometry.distance(opera)

metro[["name", "distance_deg"]].sort_values("distance_deg").head()

Small, unitless numbers — the same problem as the areas in square degrees. But distances in degrees fail in a way the areas did not make obvious: **the answer depends on which direction you walk.**

A degree of latitude is much the same length everywhere, about 111 km. A degree of longitude shrinks towards the poles by a factor of cos(latitude) — the very factor that stretches the Web Mercator map in the [first section](projections_1.ipynb). At Vienna's 48°, a degree of longitude is only about two thirds of a degree of latitude.

Two points can therefore be equally far apart on the ground and still return different "distances" in degrees. Here are two of them, each exactly one kilometre from the opera — one due east, one due north:

In [ ]:
east = Point(opera_utm.x + 1000, opera_utm.y)    # 1 km east of the opera
north = Point(opera_utm.x, opera_utm.y + 1000)   # 1 km north of the opera

pair_utm = gpd.GeoSeries([opera_utm, east, north], crs=utm_crs)
pair_deg = pair_utm.to_crs("EPSG:4326")

for label, i in [("east", 1), ("north", 2)]:
    print(f"{label:>6}: {pair_utm[0].distance(pair_utm[i]):7.1f} m"
          f" | {pair_deg[0].distance(pair_deg[i]):.6f} degrees")

In metres both come back as a kilometre, as they must. In degrees the eastward pair measures about **1.5 times** the northward one — 1 / cos(48°) once again.

A degree is not a unit of length. Treating it as one distorts not just the size of the answer but its direction, which is why no amount of rescaling can rescue a measurement made in degrees.

### 2.2. Calculating Distances — Take 2

Now the same measurement on the reprojected layers.


In [ ]:
metro_utm["distance_m"] = metro_utm.geometry.distance(opera_utm)

metro_utm[["name", "distance_m"]].sort_values("distance_m").head()

The resulting values are in metres, since the data is now in a projected UTM coordinate system. And they are readable at last: the entrances are tens of metres away, not thousandths of a degree.

The answer to the question we actually asked is the first row — but rather than reading it off by eye, let's have it printed:


In [ ]:
nearest = metro_utm.loc[metro_utm["distance_m"].idxmin()]

print(f"Nearest U-Bahn entrance: {nearest['name']}, {nearest['distance_m']:.0f} m away")

Forty-odd metres, and the entrance is called **Oper** — named after the building we measured from. That is a reassuring kind of result: when the geometry, the CRS and the arithmetic are all right, the answer tends to agree with what the data calls things.

Doing the same for every location in the district at once — nearest entrance for each — is a **spatial join**, and that is the subject of the [third module](../module_3/geoprocessing_3.ipynb).

### 2.3. Seeing What Was Measured

A column of numbers is easy to accept without looking at it. So let's draw the measurements instead — one line from the opera to each entrance, coloured by its length.

In [ ]:
distance_lines = gpd.GeoDataFrame(
    {
        "name": metro_utm["name"].values,
        "distance_m": metro_utm["distance_m"].round().values,
    },
    geometry=[LineString([opera_utm, entrance]) for entrance in metro_utm.geometry],
    crs=utm_crs,
)

distance_lines.explore(
    column="distance_m",
    cmap="viridis_r",
    tiles="cartodbpositron",
    style_kwds={"weight": 2}
)

The fan makes two things plain that the table did not.

The nearest entrances — the short pale lines — cluster right at the opera, and the lengths grow outwards to about 1.6 km at the far edge of the district. That is the shape of the numbers we printed above.

And every line runs **straight through the blocks**, across courtyards and buildings, paying no attention to where the streets go. That is precisely what `.distance()` measures, and it is what the note below is about.

> **Euclidean distance**
>
> The `.distance()` method computes **Euclidean distance** between geometries — that is, the **straight-line distance across the plane** between two points.
>
> This is sometimes referred to as **straight-line distance** or **as-the-crow-flies distance**.
>
> In practice, spatial analysis often requires **network distances** — for example, distances along roads or public transport routes.
> We will cover network analysis methods in **Module 4** of the course.
>
> That said, for many analytical tasks **Euclidean distance is a useful approximation**, particularly during exploratory spatial analysis.

> **And how good is the projected answer itself?**
>
> A projected plane is an approximation of a curved surface, so it is fair to ask what the approximation costs. The honest comparison is against a **geodesic** distance, measured on the ellipsoid itself with no projection involved — `pyproj.Geod` does that.
>
> Across the Innere Stadt the two agree to within a few centimetres. Even along a far longer line — Vienna to Salzburg, some 250 km — UTM differs from the geodesic answer by about 80 metres, or 0.03 %.
>
> Which is why UTM is the right default at the scale of this course. Where it stops being so, and what to reach for instead, is set out in the [previous section](projections_2.ipynb).


## Summary


In this section, we looked at how to **correctly measure areas and distances** using GeoPandas.

We learned:

- why geographic CRS are not suitable for spatial measurements — and that degrees distort not only the size of a distance but its direction;
- how to calculate areas using `.area`;
- how to measure distances using `.distance()`.

Choosing the right coordinate reference system is one of the most important steps in preparing data for spatial analysis.


> **Before performing spatial measurements, always check:**
>
> - whether the layer is in a **projected CRS**;
> - whether coordinates are expressed in **metres**;
> - whether all layers being used share the **same CRS**.
